# CNN for Multi-Label Classification on Triple MNIST

This notebook demonstrates a Convolutional Neural Network (CNN) for multi-label classification using the Triple MNIST dataset from Hugging Face. The goal is to predict three digits from a single image.

## 1. Setup and Data Loading

First, we import necessary libraries and load the Triple MNIST dataset directly from Hugging Face. The dataset contains images and three corresponding labels (digits).

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from datasets import load_dataset
import matplotlib.pyplot as plt

In [ ]:
# 1. Load the Dataset from Hugging Faceprint("Loading dataset...")
dataset = load_dataset("khushpatel2002/triple-mnist")

# Function to preprocess images and labels
def preprocess_data(examples):
    images = [np.array(img.convert("L")).reshape(28, 28, 1) / 255.0 for img in examples["image"]]
    labels = []
    for i in range(len(examples["label1"])): # Assuming label1, label2, label3 are present
        labels.append([examples["label1"][i], examples["label2"][i], examples["label3"][i]])
    return {"image": images, "labels": labels}

# Apply preprocessing
print("Preprocessing training data...")
processed_train_dataset = dataset["train"].map(preprocess_data, batched=True)
print("Preprocessing validation data...")
processed_val_dataset = dataset["validation"].map(preprocess_data, batched=True)
print("Preprocessing test data...")
processed_test_dataset = dataset["test"].map(preprocess_data, batched=True)

# Convert to numpy arrays
X_train = np.array(processed_train_dataset["image"])
y_train = np.array(processed_train_dataset["labels"])
X_val = np.array(processed_val_dataset["image"])
y_val = np.array(processed_val_dataset["labels"])
X_test = np.array(processed_test_dataset["image"])
y_test = np.array(processed_test_dataset["labels"])

# Convert labels to categorical (one-hot encoding) for each digit
y_train_cat = [to_categorical(y_train[:, i], num_classes=10) for i in range(3)]
y_val_cat = [to_categorical(y_val[:, i], num_classes=10) for i in range(3)]
y_test_cat = [to_categorical(y_test[:, i], num_classes=10) for i in range(3)]

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_val shape: {X_val.shape}, y_val shape: {y_val.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

## 2. Model Architecture

We define a simple CNN model with convolutional layers, batch normalization, max-pooling, and dropout. The model has three output heads, one for each digit, using `softmax` activation for multi-class classification.

In [ ]:
# 2. Define the CNN Model Architecturedef create_cnn_model(input_shape):
    input_tensor = Input(shape=input_shape)

    x = Conv2D(32, (3, 3), activation='relu', padding='same')(input_tensor)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2, 2))(x)
    x = Dropout(0.25)(x)

    x = Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2, 2))(x)
    x = Dropout(0.25)(x)

    x = Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2, 2))(x)
    x = Dropout(0.25)(x)

    x = Flatten()(x)
    x = Dense(256, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.5)(x)

    # Output layers for each digit
    output1 = Dense(10, activation='softmax', name='digit1')(x)
    output2 = Dense(10, activation='softmax', name='digit2')(x)
    output3 = Dense(10, activation='softmax', name='digit3')(x)

    model = Model(inputs=input_tensor, outputs=[output1, output2, output3])
    return model

input_shape = (28, 28, 1)
model = create_cnn_model(input_shape)
model.summary()

# Compile the model
model.compile(
    optimizer='adam',
    loss={'digit1': 'categorical_crossentropy', 'digit2': 'categorical_crossentropy', 'digit3': 'categorical_crossentropy'},
    metrics={'digit1': 'accuracy', 'digit2': 'accuracy', 'digit3': 'accuracy'}
)

## 3. Model Training

The model is compiled with `adam` optimizer and `categorical_crossentropy` loss for each output. We train the model for a few epochs.

In [ ]:
# 3. Train the Modelprint("Training model...")
history = model.fit(
    X_train, {'digit1': y_train_cat[0], 'digit2': y_train_cat[1], 'digit3': y_train_cat[2]},
    validation_data=(X_val, {'digit1': y_val_cat[0], 'digit2': y_val_cat[1], 'digit3': y_val_cat[2]}),
    epochs=10,  # Reduced epochs for quicker demonstration
    batch_size=32
)

## 4. Model Evaluation

After training, we evaluate the model's performance on the test set.

In [ ]:
# 4. Evaluate the Modelprint("Evaluating model...")
loss, digit1_loss, digit2_loss, digit3_loss, digit1_accuracy, digit2_accuracy, digit3_accuracy = model.evaluate(
    X_test, {'digit1': y_test_cat[0], 'digit2': y_test_cat[1], 'digit3': y_test_cat[2]},
    verbose=0
)

print(f"\nTest Loss: {loss:.4f}")
print(f"Test Accuracy - Digit 1: {digit1_accuracy:.4f}")
print(f"Test Accuracy - Digit 2: {digit2_accuracy:.4f}")
print(f"Test Accuracy - Digit 3: {digit3_accuracy:.4f}")

## 5. Training History Visualization

We plot the training and validation accuracy and loss for each digit over the epochs.

In [ ]:
# 5. Visualize Training Historydef plot_training_history(history):
    plt.figure(figsize=(12, 6))

    # Plot accuracy
    plt.subplot(1, 2, 1)
    plt.plot(history.history['digit1_accuracy'], label='Digit 1 Accuracy')
    plt.plot(history.history['digit2_accuracy'], label='Digit 2 Accuracy')
    plt.plot(history.history['digit3_accuracy'], label='Digit 3 Accuracy')
    plt.plot(history.history['val_digit1_accuracy'], label='Val Digit 1 Accuracy')
    plt.plot(history.history['val_digit2_accuracy'], label='Val Digit 2 Accuracy')
    plt.plot(history.history['val_digit3_accuracy'], label='Val Digit 3 Accuracy')
    plt.title('Model Accuracy')
    plt.ylabel('Accuracy')
    plt.xlabel('Epoch')
    plt.legend(loc='upper left')

    # Plot loss
    plt.subplot(1, 2, 2)
    plt.plot(history.history['digit1_loss'], label='Digit 1 Loss')
    plt.plot(history.history['digit2_loss'], label='Digit 2 Loss')
    plt.plot(history.history['digit3_loss'], label='Digit 3 Loss')
    plt.plot(history.history['val_digit1_loss'], label='Val Digit 1 Loss')
    plt.plot(history.history['val_digit2_loss'], label='Val Digit 2 Loss')
    plt.plot(history.history['val_digit3_loss'], label='Val Digit 3 Loss')
    plt.title('Model Loss')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend(loc='upper right')

    plt.tight_layout()
    plt.show()

plot_training_history(history)

## 6. Predictions and Sample Display

Finally, we make predictions on a few test images and display them along with their true and predicted labels.

In [ ]:
# 6. Make Predictions and Display Some Resultsprint("\nMaking predictions on test set...")
predictions = model.predict(X_test)

def display_predictions(images, true_labels, predicted_labels, num_samples=5):
    plt.figure(figsize=(15, 7))
    for i in range(num_samples):
        idx = np.random.randint(0, len(images))
        img = images[idx].reshape(28, 28)
        true_lbl = true_labels[idx]
        pred_lbl = [np.argmax(p[idx]) for p in predicted_labels]

        plt.subplot(1, num_samples, i + 1)
        plt.imshow(img, cmap='gray')
        plt.title(f"True: {true_lbl}\nPred: {pred_lbl}")
        plt.axis('off')
    plt.tight_layout()
    plt.show()

display_predictions(X_test, y_test, predictions, num_samples=5)